# EDA — Singapore NEA dengue clusters (bronze)

Phiên bản notebook của `scripts/eda_bronze_sg_nea.py`. Khác biệt: mỗi bước đều **in ra dữ liệu thô** trước khi rút kết luận, để có thể tự kiểm chứng thay vì tin vào con số đã tổng hợp sẵn.

Bảng nguồn: `data/bronze/sg_dengue_clusters`

In [ ]:
import sys
from pathlib import Path

# Notebook nằm trong notebooks/ nên phải thêm gốc project vào sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from ingestion.common.spark_session import build_spark_session
from ingestion.sg_nea_dengue import BRONZE_PATH

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

spark = build_spark_session("eda_notebook")
spark.sparkContext.setLogLevel("ERROR")

sdf = spark.read.format("delta").load(str(BRONZE_PATH))
print(f"{sdf.count()} dong trong bang Delta")

## 1. Nhìn toàn bộ dữ liệu thô trước

Dữ liệu còn rất nhỏ nên xem được hết — không cần lấy mẫu. Chuyển sang pandas cho dễ nhìn.

Bỏ `raw_payload` và `polygon_geojson` khỏi bảng hiển thị vì chúng dài hàng trăm ký tự; sẽ xem riêng ở dưới.

In [ ]:
df = sdf.toPandas()
df[["batch_id", "fetched_at", "object_id", "locality", "case_count", "cluster_updated_at_raw", "inc_crc"]]

## 2. Chất lượng cột

Hai câu hỏi: cột nào có null (→ ánh xạ schema sai?), và cột nào có bao nhiêu giá trị khác nhau (→ cột nào thực sự mang thông tin?).

In [ ]:
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulls": df.isna().sum(),
    "distinct": df.nunique(),
})

**Cần để ý**: `object_id` có bao nhiêu giá trị distinct so với số dòng?

Nếu bằng số cụm (chứ không bằng số dòng) thì nghĩa là OBJECTID **giữ nguyên** giữa các lần poll trong cùng chu kỳ publish. Nhưng so sánh với snapshot ngày hôm trước thì nó lại đổi — nên vẫn không dùng làm khoá xuyên thời gian được.

In [ ]:
# Cung mot locality, OBJECTID co doi giua cac lo khong?
df.pivot_table(index="locality", columns="batch_id", values="object_id", aggfunc="first")

## 3. Nhịp poll của chúng ta

Đây là thuộc tính của scheduler, **không phải** của dữ liệu. Đừng nhầm hai thứ.

In [ ]:
polls = (
    df.groupby("batch_id")
    .agg(rows=("object_id", "size"), fetched_at=("fetched_at", "min"))
    .sort_values("fetched_at")
)
polls["gap_hours"] = polls["fetched_at"].diff().dt.total_seconds() / 3600
polls

## 4. Nhịp publish của NEA

Đây mới là con số nên dùng để quyết định nhịp poll. Suy ra từ `FMEL_UPD_D` — thời điểm NEA sửa từng cụm lần cuối.

Trường này trải dài nhiều ngày nên tuy ta mới poll 2 lần, nó vẫn cho thấy được nhịp thật của nguồn.

In [ ]:
updates = (
    df.assign(updated_at=pd.to_datetime(df["cluster_updated_at_raw"], format="%Y%m%d%H%M%S"))
    .groupby("updated_at")["locality"]
    .nunique()
    .to_frame("clusters")
    .sort_index()
)
updates["gap_days"] = updates.index.to_series().diff().dt.total_seconds() / 86400
updates["time_of_day"] = updates.index.strftime("%H:%M")
updates

**Nhìn cột `time_of_day`**: các mốc có tụm lại quanh một giờ nhất định không?

Nếu có, đó là dấu hiệu NEA chạy một tác vụ theo lịch hằng ngày — và giờ đó còn gợi ý múi giờ của trường này (nó không mang thông tin timezone). Publish lúc 15:00 giờ Singapore là hợp lý; lúc 23:00 giờ Singapore thì khó tin.

## 5. Phân bố số ca

Lấy lô mới nhất làm trạng thái hiện tại.

In [ ]:
latest = df[df["batch_id"] == df["batch_id"].max()]
latest[["locality", "case_count", "cluster_updated_at_raw"]].sort_values("case_count", ascending=False)

In [ ]:
cases = latest["case_count"]
print(cases.describe())
print()
print(f"tong so ca      : {cases.sum()}")
print(f"trung binh      : {cases.mean():.1f}")
print(f"trung vi        : {cases.median():.1f}")
print(f"cum lon nhat chiem: {cases.max() / cases.sum():.0%} tong so ca")

**So sánh trung bình với trung vị.** Nếu chúng lệch nhau nhiều thì phân bố bị kéo bởi vài giá trị cực lớn — và trung bình trở thành con số gây hiểu nhầm (sẽ không có cụm nào thật sự mang giá trị đó).

Điều này ảnh hưởng trực tiếp tới cách tính chỉ số rủi ro cho trang cảnh báo sau này.

## 6. Có gì thay đổi giữa các lô?

Ghép theo `locality` chứ **không** ghép theo `object_id` — lý do đã thấy ở mục 2.

`inc_crc` là checksum nội dung của chính NEA: đổi giá trị nghĩa là cụm đó có thay đổi thật.

In [ ]:
crc = df.pivot_table(index="locality", columns="batch_id", values="inc_crc", aggfunc="first")
batches = sorted(df["batch_id"].unique())

if len(batches) >= 2:
    first, last = batches[0], batches[-1]
    crc["changed"] = crc[first].notna() & crc[last].notna() & (crc[first] != crc[last])
    print(f"So sanh {first} -> {last}")
    print(f"  khong doi : {(~crc['changed'] & crc[first].notna() & crc[last].notna()).sum()}")
    print(f"  co doi    : {crc['changed'].sum()}")
    print(f"  moi xuat hien: {crc[first].isna().sum()}")
    print(f"  bien mat  : {crc[last].isna().sum()}")
else:
    print("Can it nhat 2 lo de so sanh")

crc

## 7. Xem một bản ghi gốc

`raw_payload` giữ nguyên feature gốc từ API. Đây là lý do Bronze không sợ mất dữ liệu khi NEA thêm field mới — kiểm chứng bằng cách so sánh các key trong payload với các cột đang có.

In [ ]:
import json

sample = json.loads(latest.iloc[0]["raw_payload"])
print("Cac key trong properties:")
for key, value in sample["properties"].items():
    mapped = "-> co cot rieng" if key in {"OBJECTID", "LOCALITY", "CASE_SIZE", "FMEL_UPD_D", "INC_CRC"} else "   chi nam trong raw_payload"
    print(f"  {key:<20} {str(value)[:40]:<42} {mapped}")

print(f"\ngeometry: {sample['geometry']['type']}, {len(sample['geometry']['coordinates'][0])} diem")

## Giới hạn của phân tích này

Mẫu hiện tại quá nhỏ để kết luận chắc chắn:

- Chỉ có vài lô, trong phạm vi 1 ngày → nhận định về nhịp poll chưa có giá trị
- `FMEL_UPD_D` trải vài ngày → nhận định về nhịp publish của NEA đáng tin hơn, nhưng vẫn chỉ dựa trên 4 mốc
- Múi giờ của `FMEL_UPD_D` mới là **suy luận**, chưa xác minh được
- Giả thuyết "NEA không công bố cụm dưới 2 ca" mới chỉ dựa trên `min = 2`

Chạy lại notebook này sau khi thu thập ≥2 tuần dữ liệu.

In [ ]:
spark.stop()